In [ ]:
# @title
!pip install bert-score
!pip install datasets
!pip install sentence-transformers datasets accelerate transformers wandb
!pip install sentence-transformers torch sklearn bert-score
!pip install sentencepiece
!pip install -q sentencepiece transformers sentence-transformers bert-score torch scikit-learn pandas numpy
!pip install -q pytorch-crf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


Post-Segmentation Normalization (FEMSeg_kaz+MAPP)

In [ ]:
# @title
# ============================================================
# FULL QA PIPELINE BY SCHEMA:
# Question (Kazakh)
#   -> [1] Morphological Segmentation (FEMSeg)
#   -> [2] Morphology-Aware Normalization (MAPP)
#   -> [3] Hybrid Retrieval: Dense (MiniLM) + Morphological Similarity
#   -> [4] Reranking (LLM-lite semantic reranker)
#   -> [5] Answer Post-processing (MAPP generation)
#   -> Final Answer
#
# HELD-OUT TEST, FAIR EVALUATION, NO FINE-TUNING
#
# Metrics:
#   Exact@1
#   TokenF1@1
#   MeanCos@1(QSim)
#   Semantic@1(ans_cos>=0.85)
#   BERTScoreF1@1
# ============================================================

# =========================
# ENV
# =========================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_DISABLED"] = "true"

import re
import glob
import time
import json
import random
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Set
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from bert_score import score as bert_score
except Exception:
    bert_score = None

try:
    from torchcrf import CRF
except Exception:
    CRF = None

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False


# --------------------------
# Safety checks
# --------------------------
if CRF is None:
    raise ImportError(
        "torchcrf табылмады. Colab-та мына команданы орындаңыз:\n"
        "!pip install torchcrf"
    )


# --------------------------
# Determinism
# --------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

try:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥 DEVICE:", DEVICE)


# --------------------------
# CONFIG
# --------------------------
DATA_JSON_PATH = "50k.json"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

TEST_SIZE = 0.10
SEM_THR = 0.85

ENCODE_BATCH_SIZE = 64
SIM_BATCH_SIZE = 256
PRINT_PROGRESS_EVERY = 5000

EXPORT_CSV = True
CSV_PATH = "qa_schema_hybrid_rerank_results.csv"

# [3] Hybrid retrieval weights
ALPHA_SEG = 0.40                  # dense dual-view ішінде segmented view салмағы
HYBRID_DENSE_WEIGHT = 0.72        # dense retrieval салмағы
HYBRID_MORPH_WEIGHT = 0.28        # morphological similarity салмағы
DENSE_TOPK = 80                   # dense shortlist

# [4] Reranking weights (LLM-lite)
RERANK_TOPK = 12
RERANK_HYBRID_WEIGHT = 0.75
RERANK_ANS_WEIGHT = 0.25

# [2] MAPP config
USE_MAPP = True
MAPP_TAG = "[MAPP]"
MAPP_MIN_WORD_LEN = 4

# [5] Post-processing config
POSTPROCESS_FOR_EVAL = True


# ============================================================
# 0) Robust JSON loading
# ============================================================

def find_data_path(p: str) -> str:
    if Path(p).exists():
        return p
    candidates = [f"/content/{p}", f"/content/drive/MyDrive/{p}"]
    for c in candidates:
        if Path(c).exists():
            return c
    name = Path(p).name
    hits = glob.glob(f"**/{name}", recursive=True)
    if hits:
        return hits[0]
    near = glob.glob("**/*.json", recursive=True)
    raise FileNotFoundError(
        f"❌ File not found: {p}\nPWD: {Path.cwd()}\n"
        f"Found .json (first 30):\n" + "\n".join(near[:30])
    )

def _fix_invalid_backslashes(text: str) -> str:
    return re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)

def _remove_trailing_commas(text: str) -> str:
    return re.sub(r",(\s*[}\]])", r"\1", text)

def _normalize_json_text(text: str) -> str:
    text = text.replace("\ufeff", "").strip()
    text = _remove_trailing_commas(text)
    text = _fix_invalid_backslashes(text)
    return text

def _normalize_record(rec: Any) -> Optional[Dict[str, str]]:
    if not isinstance(rec, dict):
        return None
    if "question" in rec and "answer" in rec:
        q = str(rec["question"]).strip()
        a = str(rec["answer"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    if "instruction" in rec and "response" in rec:
        q = str(rec["instruction"]).strip()
        a = str(rec["response"]).strip()
        if q and a:
            return {"question": q, "answer": a}
    return None

def _extract_records(obj: Any) -> List[Dict[str, str]]:
    normalized = _normalize_record(obj)
    if normalized is not None:
        return [normalized]

    if isinstance(obj, list):
        out = []
        for item in obj:
            norm = _normalize_record(item)
            if norm is not None:
                out.append(norm)
        return out

    if isinstance(obj, dict):
        for key in ("data", "items", "qa_data", "records", "dataset"):
            if key in obj and isinstance(obj[key], list):
                out = []
                for item in obj[key]:
                    norm = _normalize_record(item)
                    if norm is not None:
                        out.append(norm)
                return out
    return []

def _parse_as_single_json(text: str) -> List[Dict[str, str]]:
    obj = json.loads(_normalize_json_text(text))
    records = _extract_records(obj)
    if not records:
        raise ValueError("Single JSON ішінде QA жазба табылмады.")
    return records

def _parse_as_multiple_json_objects(text: str) -> List[Dict[str, str]]:
    text = _normalize_json_text(text)
    decoder = json.JSONDecoder()
    idx = 0
    records = []

    while idx < len(text):
        while idx < len(text) and text[idx] in " \r\n\t,":
            idx += 1
        if idx >= len(text):
            break

        obj, end = decoder.raw_decode(text, idx)
        idx = end
        extracted = _extract_records(obj)
        if extracted:
            records.extend(extracted)

    if not records:
        raise ValueError("Concatenated JSON objects ішінде QA жазба табылмады.")
    return records

def _parse_line_by_line(text: str) -> List[Dict[str, str]]:
    records = []
    bad_lines = []

    for line_no, line in enumerate(text.splitlines(), start=1):
        s = line.strip()
        if not s:
            continue
        if s.endswith(","):
            s = s[:-1].strip()
        s = _normalize_json_text(s)

        try:
            obj = json.loads(s)
            extracted = _extract_records(obj)
            if extracted:
                records.extend(extracted)
            else:
                bad_lines.append((line_no, "QA өрістері табылмады", s[:200]))
        except Exception as e:
            bad_lines.append((line_no, str(e), s[:200]))

    if not records:
        preview = "\n".join(
            [f"line {ln}: {err} | {sample}" for ln, err, sample in bad_lines[:5]]
        )
        raise ValueError(
            "Файлдан бірде-бір QA жазба оқылмады.\n"
            f"Алғашқы қате жолдар:\n{preview}"
        )

    if bad_lines:
        print(f"⚠ Ескерту: {len(bad_lines)} жол оқылмады, бірақ қалғандары жүктелді.")

    return records

def load_qa_json_robust(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8-sig", errors="ignore")
    errors = []

    for parser in (
        _parse_as_single_json,
        _parse_as_multiple_json_objects,
        _parse_line_by_line,
    ):
        try:
            records = parser(text)
            if records:
                return records
        except Exception as e:
            errors.append(f"{parser.__name__}: {e}")

    raise ValueError("JSON файлын оқу мүмкін болмады.\n" + "\n".join(errors))


data_path = find_data_path(DATA_JSON_PATH)
print(f"📥 QA JSON файлдан жүктеу: {data_path}")

data = load_qa_json_robust(data_path)
qa_data = pd.DataFrame(data)

assert {"question", "answer"}.issubset(qa_data.columns), \
    "JSON-да question/answer немесе instruction/response болуы тиіс."

qa_data["question"] = qa_data["question"].astype(str).str.strip()
qa_data["answer"]   = qa_data["answer"].astype(str).str.strip()
qa_data = qa_data[(qa_data["question"] != "") & (qa_data["answer"] != "")].reset_index(drop=True)

print(f"📚 Барлық QA жазба саны: {len(qa_data)}")
print(qa_data.head(3))


# ============================================================
# 1) Base text normalization
# ============================================================

_punct_space_left  = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space       = re.compile(r"\s+")

def clean_view(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t

def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", clean_view(text).lower()).strip()

def tokens(text: str) -> List[str]:
    t = clean_view(text).lower()
    return re.findall(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+", t)

def token_f1(pred: str, gold: str) -> float:
    p = tokens(pred)
    g = tokens(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0

    pc = Counter(p)
    gc = Counter(g)
    inter = sum((pc & gc).values())
    if inter == 0:
        return 0.0

    prec = inter / max(1, len(p))
    rec  = inter / max(1, len(g))
    return (2 * prec * rec) / (prec + rec + 1e-12)

def cosine01(x: float) -> float:
    return max(0.0, min(1.0, (x + 1.0) * 0.5))


# ============================================================
# [1] FEMSeg_v3 (Morphological Segmentation)
# ============================================================

BMES_TAGS = ["B", "M", "E", "S"]
TAG2ID = {t: i for i, t in enumerate(BMES_TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}

class FEMSegV3(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int = 128,
        lstm_hidden_dim: int = 256,
        dropout: float = 0.3,
        num_tags: int = len(BMES_TAGS),
    ):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=char_emb_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(lstm_hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward_features(self, x, mask):
        emb = self.emb(x)
        emb = self.dropout(emb)
        h, _ = self.lstm(emb)
        h = self.dropout(h)
        return h

    def forward_logits(self, x, mask):
        feats = self.forward_features(x, mask)
        return self.fc(feats)

    def forward(self, x, tags=None, mask=None):
        if mask is None:
            mask = x != 0
        emissions = self.forward_logits(x, mask)
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)

def bmes_to_cse(chars: List[str], tags: List[str]) -> str:
    morphs: List[str] = []
    cur = ""
    for ch, t in zip(chars, tags):
        if t == "B":
            if cur:
                morphs.append(cur)
            cur = ch
        elif t == "M":
            cur += ch
        elif t == "E":
            cur += ch
            morphs.append(cur)
            cur = ""
        elif t == "S":
            if cur:
                morphs.append(cur)
            morphs.append(ch)
            cur = ""
    if cur:
        morphs.append(cur)
    return "@@ ".join(morphs)

class KazMorphSegmentor:
    def __init__(self, char2id_path: str, ckpt_path: str, use_cuda: bool = True):
        with open(char2id_path, "r", encoding="utf-8") as f:
            self.char2id: Dict[str, int] = json.load(f)

        self.pad_id = 0
        self.unk_id = self.char2id.get("<unk>") or self.char2id.get("[UNK]") or 1
        self.device = torch.device(
            "cuda" if (use_cuda and torch.cuda.is_available()) else "cpu"
        )

        vocab_size = max(self.char2id.values()) + 1
        self.model = FEMSegV3(vocab_size=vocab_size)
        self.model.to(self.device)

        state = torch.load(ckpt_path, map_location=self.device)
        if isinstance(state, dict):
            if "state_dict" in state:
                state = state["state_dict"]
            elif "model_state_dict" in state:
                state = state["model_state_dict"]

        if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}

        self.model.load_state_dict(state, strict=True)
        self.model.eval()

        self.word_cache: Dict[str, str] = {}
        self.text_cache: Dict[str, str] = {}

    def _word_to_tensor(self, word: str):
        ids = [self.char2id.get(ch, self.unk_id) for ch in list(word)]
        x = torch.tensor([ids], dtype=torch.long, device=self.device)
        mask = (x != self.pad_id)
        return x, mask

    def segment_word(self, word: str) -> str:
        word = word.strip()
        if not word:
            return ""

        cached = self.word_cache.get(word)
        if cached is not None:
            return cached

        x, mask = self._word_to_tensor(word)
        with torch.no_grad():
            best_paths = self.model(x, mask=mask)
        tags = [ID2TAG[i] for i in best_paths[0]]
        result = bmes_to_cse(list(word), tags)
        self.word_cache[word] = result
        return result

    def segment_text(self, text: str) -> str:
        text = text.strip()
        if not text:
            return ""

        cached = self.text_cache.get(text)
        if cached is not None:
            return cached

        words = text.split()
        out_words = [self.segment_word(w) for w in words]
        result = " | ".join(out_words)
        self.text_cache[text] = result
        return result

def femseg_to_sp(text: str) -> str:
    """
    FEMSeg нәтижесін SentencePiece-like форматқа ауыстыру.
    """
    text = clean_view(str(text))
    if not text:
        return ""

    seg_line = femseg.segment_text(text)
    sp_tokens = []

    for word_seg in seg_line.split(" | "):
        word_seg = word_seg.strip()
        if not word_seg:
            continue
        morphs = [m.strip() for m in word_seg.split("@@ ") if m.strip()]
        if not morphs:
            continue
        sp_tokens.append("▁" + morphs[0])
        sp_tokens.extend(morphs[1:])

    return " ".join(sp_tokens)

def sp_token_set(sp_text: str) -> Set[str]:
    out = set()
    for tok in str(sp_text).split():
        tok = tok.replace("▁", "").strip()
        if tok:
            out.add(tok)
    return out


# ============================================================
# FEMSeg paths
# ============================================================
if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)

FEMSEG_CHAR2ID = "/content/drive/MyDrive/KAZ_MORPH/char2id_femseg_v3_50k.json"
FEMSEG_CKPT    = "/content/drive/MyDrive/KAZ_MORPH/femseg_v3_50k_epoch3.pt"

assert os.path.exists(FEMSEG_CHAR2ID), f"char2id_femseg_v3_50k.json табылмады: {FEMSEG_CHAR2ID}"
assert os.path.exists(FEMSEG_CKPT),    f"femseg_v3_50k_epoch3.pt табылмады: {FEMSEG_CKPT}"

print("⬇️ FEMSeg_v3 сегментаторын жүктеу...")
femseg = KazMorphSegmentor(
    char2id_path=FEMSEG_CHAR2ID,
    ckpt_path=FEMSEG_CKPT,
    use_cuda=True,
)
print("✅ FEMSeg_v3 дайын.")
print("🔎 FEMSeg тест:", femseg_to_sp("Мен мектепке барамын"))


# ============================================================
# [2] MAPP — Morphology-Aware Normalization
# ============================================================

WORD_RE = re.compile(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9_+#-]+", re.UNICODE)

PROTECTED_WORDS = {
    "не", "неге", "неліктен", "қалай", "қайда", "қайдан", "қашан",
    "қандай", "қанша", "қай", "кім", "кімге", "кімнің", "неше",
    "python", "def", "for", "while", "if", "elif", "else", "print", "input",
    "list", "dict", "tuple", "set", "int", "str", "float", "bool", "none",
    "true", "false", "class", "return", "break", "continue", "range", "len",
    "append", "insert", "remove", "pop", "sort", "lambda", "map", "filter",
    "reduce", "zip", "enumerate", "and", "or", "not", "in", "is"
}

MAPP_WORD_MAP = {
    "калай": "қалай",
    "кандай": "қандай",
    "кайда": "қайда",
    "кашан": "қашан",
    "канша": "қанша",
    "аныкта": "анықта",
    "аныктайды": "анықтайды",
    "аныктаймыз": "анықтаймыз",
    "есептеймз": "есептейміз",
    "пайтон": "python",
    "питон": "python",
    "функсия": "функция",
    "функсиа": "функция",
    "циклд": "цикл",
    "стр": "str",
    "инт": "int",
    "бул": "bool",
    "принт": "print",
    "инпут": "input",
}

_mapp_word_cache: Dict[str, str] = {}
_mapp_text_cache: Dict[str, str] = {}

def _basic_mapp_word_norm(word: str) -> str:
    w = str(word).strip().lower()
    return MAPP_WORD_MAP.get(w, w)

def _is_ascii_programming_token(word: str) -> bool:
    return bool(re.fullmatch(r"[a-z0-9_+#-]+", word))

def _split_seg_word(seg_word: str) -> List[str]:
    return [m.strip() for m in str(seg_word).split("@@ ") if m.strip()]

def _mapp_stem_from_femseg(word: str) -> str:
    """
    Агрессивті емес MAPP:
    - protected / қысқа / programming token болса, сол күйі қалады
    - әйтпесе FEMSeg арқылы бірінші негізгі морф (stem) алынады
    """
    w = _basic_mapp_word_norm(word)

    if not USE_MAPP:
        return w
    if not w:
        return w

    cached = _mapp_word_cache.get(w)
    if cached is not None:
        return cached

    if w in PROTECTED_WORDS:
        _mapp_word_cache[w] = w
        return w
    if len(w) < MAPP_MIN_WORD_LEN:
        _mapp_word_cache[w] = w
        return w
    if w.isdigit():
        _mapp_word_cache[w] = w
        return w
    if _is_ascii_programming_token(w):
        _mapp_word_cache[w] = w
        return w

    try:
        seg_word = femseg.segment_word(w)
        morphs = _split_seg_word(seg_word)
        if not morphs:
            result = w
        else:
            stem = _basic_mapp_word_norm(morphs[0])
            result = stem if len(stem) >= 2 else w
    except Exception:
        result = w

    _mapp_word_cache[w] = result
    return result

def mapp_normalize(text: str) -> str:
    """
    MAPP:
    - clean_view
    - lower
    - ortho fixes
    - FEMSeg-based stem canonicalization
    """
    t = clean_view(text).lower().strip()
    if not t:
        return ""

    cached = _mapp_text_cache.get(t)
    if cached is not None:
        return cached

    words = WORD_RE.findall(t)
    out = []
    prev = None

    for w in words:
        nw = _mapp_stem_from_femseg(w)
        if nw and nw != prev:
            out.append(nw)
            prev = nw

    result = " ".join(out).strip()
    if not result:
        result = t

    _mapp_text_cache[t] = result
    return result

def mapp_token_set(text: str) -> Set[str]:
    return set(WORD_RE.findall(mapp_normalize(text)))

def compose_dense_view(q_clean: str, q_mapp: str) -> str:
    q_clean = clean_view(q_clean)
    q_mapp  = clean_view(q_mapp)

    if not q_mapp:
        return q_clean
    if norm_for_exact(q_clean) == norm_for_exact(q_mapp):
        return q_clean

    return f"{q_clean} {MAPP_TAG} {q_mapp}"


# ============================================================
# [5] Answer Post-processing (MAPP generation)
# ============================================================

POST_WORD_MAP = {
    "пайтон": "Python",
    "питон": "Python",
    "функсиа": "функция",
    "функсия": "функция",
    "принт": "print",
    "инпут": "input",
}

def _is_code_like_line(line: str) -> bool:
    s = line.rstrip()
    if not s.strip():
        return False
    if s.startswith("    ") or s.startswith("\t"):
        return True
    code_heads = ("def ", "for ", "while ", "if ", "elif ", "else:", "return ", "class ", "print(")
    if s.lstrip().startswith(code_heads):
        return True
    if re.match(r"^\s*#.*$", s):
        return True
    return False

def _replace_whole_words(text: str, mapping: Dict[str, str]) -> str:
    out = text
    for a, b in mapping.items():
        out = re.sub(rf"\b{re.escape(a)}\b", b, out, flags=re.IGNORECASE)
    return out

def answer_postprocess(text: str) -> str:
    """
    Консервативті post-processing:
    - артық бос орынды түзету
    - жиі термин қатесін стандарттау
    - қайталанатын бос/бірдей жолдарды қысқарту
    - код жолдарын бұзбау
    """
    if text is None:
        return ""

    t = str(text).replace("\r\n", "\n").strip()
    if not t:
        return ""

    lines = t.split("\n")
    out_lines = []
    prev_norm = None

    for line in lines:
        raw = line.rstrip()

        if not raw.strip():
            if out_lines and out_lines[-1] != "":
                out_lines.append("")
            prev_norm = None
            continue

        if _is_code_like_line(raw):
            cleaned = raw.rstrip()
        else:
            cleaned = clean_view(raw)
            cleaned = _replace_whole_words(cleaned, POST_WORD_MAP)

        normed = cleaned.strip()
        if normed != prev_norm:
            out_lines.append(cleaned)
            prev_norm = normed

    result = "\n".join(out_lines).strip()
    result = re.sub(r"\n{3,}", "\n\n", result).strip()
    return result


# ============================================================
# Load encoder
# ============================================================

print("⬇️ MiniLM encoder-ін жүктеу:", MODEL_NAME)
retr_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
retr_model.eval()
print("✅ MiniLM моделі дайын.")


# ============================================================
# Build all views
# ============================================================

qa_data["q_clean"] = qa_data["question"].map(clean_view)
qa_data["a_clean"] = qa_data["answer"].map(clean_view)

print(f"\n🧠 MAPP нормализация басталды... ({len(qa_data)} жазба)")
t0 = time.time()

unique_questions_clean = qa_data["q_clean"].unique().tolist()
q_mapp_map = {}

for i, txt in enumerate(unique_questions_clean, start=1):
    q_mapp_map[txt] = mapp_normalize(txt)
    if i % PRINT_PROGRESS_EVERY == 0 or i == len(unique_questions_clean):
        print(f"   MAPP {i}/{len(unique_questions_clean)} дайын | {time.time() - t0:.1f} сек")

qa_data["q_mapp"]  = qa_data["q_clean"].map(q_mapp_map)
qa_data["q_dense"] = qa_data.apply(
    lambda r: compose_dense_view(r["q_clean"], r["q_mapp"]), axis=1
)

print(f"\n🧩 FEMSeg сегментация басталды... ({len(qa_data)} жазба)")
t0 = time.time()

unique_questions_mapp = qa_data["q_mapp"].unique().tolist()
q_sp_map = {}

for i, txt in enumerate(unique_questions_mapp, start=1):
    q_sp_map[txt] = femseg_to_sp(txt)
    if i % PRINT_PROGRESS_EVERY == 0 or i == len(unique_questions_mapp):
        print(f"   SEG {i}/{len(unique_questions_mapp)} дайын | {time.time() - t0:.1f} сек")

qa_data["q_sp"] = qa_data["q_mapp"].map(q_sp_map)

# Answer final text
qa_data["a_final"] = qa_data["a_clean"].map(answer_postprocess)

print("\n🔎 Мысал:")
print(qa_data[["question", "q_clean", "q_mapp", "q_dense", "q_sp", "a_final"]].head())


# ============================================================
# Split
# ============================================================

train_df, test_df = train_test_split(
    qa_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_idx = train_df.index.to_numpy()
test_idx  = test_df.index.to_numpy()

print(f"\n📊 TRAIN: {len(train_df)} | TEST: {len(test_df)} | seed={SEED} | test_size={TEST_SIZE}")

train_questions = train_df["question"].tolist()
train_answers   = train_df["a_final"].tolist()
test_questions  = test_df["question"].tolist()
test_answers    = test_df["a_final"].tolist()


# ============================================================
# Embedding helpers
# ============================================================

def _encode_dual(
    dense_texts: List[str],
    seg_texts: List[str],
    batch_size: int = ENCODE_BATCH_SIZE,
    normalize: bool = True,
) -> np.ndarray:
    """
    Dense dual-view:
      final = ALPHA_SEG * seg_emb + (1 - ALPHA_SEG) * dense_emb
    dense_emb = clean + MAPP info
    seg_emb   = FEMSeg view
    """
    dense_texts = [str(t) for t in dense_texts]
    seg_texts   = [str(t) for t in seg_texts]

    v_seg = retr_model.encode(
        seg_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )

    v_dense = retr_model.encode(
        dense_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )

    vecs = ALPHA_SEG * v_seg + (1.0 - ALPHA_SEG) * v_dense

    if normalize:
        norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
        vecs = vecs / norms

    return vecs.astype(np.float32)

def _encode_answers(
    texts: List[str],
    batch_size: int = ENCODE_BATCH_SIZE,
    normalize: bool = True,
) -> np.ndarray:
    clean = [answer_postprocess(t) for t in texts]
    vecs = retr_model.encode(
        clean,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
        show_progress_bar=False,
    )
    return vecs.astype(np.float32)

def _build_query_views(text: str):
    q_clean = clean_view(text)
    q_mapp  = mapp_normalize(q_clean)
    q_dense = compose_dense_view(q_clean, q_mapp)
    q_sp    = femseg_to_sp(q_mapp)
    q_tok_set = set(WORD_RE.findall(q_mapp))
    q_seg_set = sp_token_set(q_sp)
    return q_clean, q_mapp, q_dense, q_sp, q_tok_set, q_seg_set


# ============================================================
# Precompute full arrays
# ============================================================

qa_q_dense_full = qa_data["q_dense"].tolist()
qa_q_sp_full    = qa_data["q_sp"].tolist()
qa_a_final_full = qa_data["a_final"].tolist()
qa_q_mapp_full  = qa_data["q_mapp"].tolist()

print("\n📦 FULL QA question embedding-тері есептелуде...")
t0 = time.time()
qa_q_emb_full = _encode_dual(
    dense_texts=qa_q_dense_full,
    seg_texts=qa_q_sp_full,
    batch_size=ENCODE_BATCH_SIZE,
    normalize=True,
)
print(f"✅ FULL QA question embedding дайын: {time.time() - t0:.2f} сек")

print("\n📦 FULL QA answer embedding-тері есептелуде...")
t0 = time.time()
qa_a_emb_full = _encode_answers(
    qa_a_final_full,
    batch_size=ENCODE_BATCH_SIZE,
    normalize=True,
)
print(f"✅ FULL QA answer embedding дайын: {time.time() - t0:.2f} сек")

# Morph token sets
qa_q_tok_sets_full = [set(WORD_RE.findall(x)) for x in qa_q_mapp_full]
qa_q_seg_sets_full = [sp_token_set(x) for x in qa_q_sp_full]

# Split arrays
train_q_emb = qa_q_emb_full[train_idx]
test_q_emb  = qa_q_emb_full[test_idx]

train_a_emb = qa_a_emb_full[train_idx]
test_a_emb  = qa_a_emb_full[test_idx]

train_q_tok_sets = [qa_q_tok_sets_full[i] for i in train_idx]
test_q_tok_sets  = [qa_q_tok_sets_full[i] for i in test_idx]

train_q_seg_sets = [qa_q_seg_sets_full[i] for i in train_idx]
test_q_seg_sets  = [qa_q_seg_sets_full[i] for i in test_idx]


# ============================================================
# [3] Hybrid Retrieval helpers
# ============================================================

def jaccard_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))

def dice_sim(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return (2.0 * len(a & b)) / max(1.0, len(a) + len(b))

def morph_similarity(
    q_tok_set: Set[str],
    c_tok_set: Set[str],
    q_seg_set: Set[str],
    c_seg_set: Set[str],
) -> float:
    """
    Morph similarity:
    - MAPP token overlap
    - FEMSeg segment overlap
    """
    token_score = jaccard_sim(q_tok_set, c_tok_set)
    seg_score   = dice_sim(q_seg_set, c_seg_set)
    return 0.65 * token_score + 0.35 * seg_score

def get_topk_sorted(scores: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    k = min(k, scores.shape[0])
    if k <= 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    idx = np.argpartition(-scores, kth=k - 1)[:k]
    vals = scores[idx]
    order = np.argsort(-vals)
    idx = idx[order]
    vals = vals[order]
    return idx.astype(np.int32), vals.astype(np.float32)


# ============================================================
# [4] LLM-lite Reranking helpers
# ============================================================

def rerank_candidates_lite(
    q_emb: np.ndarray,
    q_tok_set: Set[str],
    q_seg_set: Set[str],
    cand_indices: List[int],
    cand_dense_scores: List[float],
    kb_q_tok_sets: List[Set[str]],
    kb_q_seg_sets: List[Set[str]],
    kb_a_emb: np.ndarray,
):
    """
    Reranking:
    - hybrid retrieval score
    - query-answer semantic relevance
    """
    hybrid_rows = []

    for idx, dscore in zip(cand_indices, cand_dense_scores):
        morph = morph_similarity(
            q_tok_set=q_tok_set,
            c_tok_set=kb_q_tok_sets[idx],
            q_seg_set=q_seg_set,
            c_seg_set=kb_q_seg_sets[idx],
        )

        dense_part = cosine01(float(dscore))
        hybrid = HYBRID_DENSE_WEIGHT * dense_part + HYBRID_MORPH_WEIGHT * morph

        hybrid_rows.append((idx, float(dscore), float(morph), float(hybrid)))

    hybrid_rows.sort(key=lambda x: x[3], reverse=True)
    hybrid_rows = hybrid_rows[:min(RERANK_TOPK, len(hybrid_rows))]

    best = None
    qv = q_emb.reshape(-1)

    for idx, dscore, morph, hybrid in hybrid_rows:
        ans_rel = float(np.dot(qv, kb_a_emb[idx]))
        ans_rel01 = cosine01(ans_rel)

        rerank_score = (
            RERANK_HYBRID_WEIGHT * hybrid +
            RERANK_ANS_WEIGHT * ans_rel01
        )

        row = {
            "idx": idx,
            "dense_qsim": float(dscore),
            "morph_sim": float(morph),
            "hybrid_score": float(hybrid),
            "ans_rel": float(ans_rel),
            "rerank_score": float(rerank_score),
        }

        if best is None or row["rerank_score"] > best["rerank_score"]:
            best = row

    return best


# ============================================================
# Retrieval runtime / eval
# ============================================================

_query_emb_cache: Dict[str, np.ndarray] = {}

def _get_query_embedding_from_views(q_dense: str, q_sp: str) -> np.ndarray:
    cache_key = f"{q_dense} ||| {q_sp}"
    cached = _query_emb_cache.get(cache_key)
    if cached is not None:
        return cached

    qv = _encode_dual(
        dense_texts=[q_dense],
        seg_texts=[q_sp],
        batch_size=1,
        normalize=True,
    )
    _query_emb_cache[cache_key] = qv
    return qv

def _retrieve_from_kb(
    question: str,
    kb_q_emb: np.ndarray,
    kb_a_emb: np.ndarray,
    kb_answers: List[str],
    kb_q_tok_sets: List[Set[str]],
    kb_q_seg_sets: List[Set[str]],
    threshold: float = 0.0,
):
    q_clean, q_mapp, q_dense, q_sp, q_tok_set, q_seg_set = _build_query_views(question)
    qv = _get_query_embedding_from_views(q_dense, q_sp)

    sims = (kb_q_emb @ qv.T).squeeze(1)
    top_idx, top_vals = get_topk_sorted(sims, DENSE_TOPK)

    best = rerank_candidates_lite(
        q_emb=qv.squeeze(0),
        q_tok_set=q_tok_set,
        q_seg_set=q_seg_set,
        cand_indices=top_idx.tolist(),
        cand_dense_scores=top_vals.tolist(),
        kb_q_tok_sets=kb_q_tok_sets,
        kb_q_seg_sets=kb_q_seg_sets,
        kb_a_emb=kb_a_emb,
    )

    if best is None:
        return "Кешіріңіз, нақты жауап табылмады.", -1, 0.0, 0.0, 0.0, q_clean, q_mapp, q_dense

    final_idx = int(best["idx"])
    final_dense_qsim = float(best["dense_qsim"])

    if final_dense_qsim < threshold:
        return "Кешіріңіз, нақты жауап табылмады.", -1, final_dense_qsim, best["morph_sim"], best["rerank_score"], q_clean, q_mapp, q_dense

    final_answer = kb_answers[final_idx]
    if POSTPROCESS_FOR_EVAL:
        final_answer = answer_postprocess(final_answer)

    return (
        final_answer,
        final_idx,
        final_dense_qsim,
        float(best["morph_sim"]),
        float(best["rerank_score"]),
        q_clean,
        q_mapp,
        q_dense,
    )

def ask_question_runtime(question: str, threshold: float = 0.0):
    return _retrieve_from_kb(
        question=question,
        kb_q_emb=qa_q_emb_full,
        kb_a_emb=qa_a_emb_full,
        kb_answers=qa_a_final_full,
        kb_q_tok_sets=qa_q_tok_sets_full,
        kb_q_seg_sets=qa_q_seg_sets_full,
        threshold=threshold,
    )

def ask_question_eval(question: str):
    return _retrieve_from_kb(
        question=question,
        kb_q_emb=train_q_emb,
        kb_a_emb=train_a_emb,
        kb_answers=train_answers,
        kb_q_tok_sets=train_q_tok_sets,
        kb_q_seg_sets=train_q_seg_sets,
        threshold=0.0,
    )


# ============================================================
# BERTScore helper
# ============================================================

def _bert_lang_try(preds: List[str], golds: List[str]) -> Optional[float]:
    if bert_score is None:
        return None
    for lang in ("kk", "tr", "en"):
        try:
            _, _, F1 = bert_score(preds, golds, lang=lang, rescale_with_baseline=True)
            arr = F1.numpy() if hasattr(F1, "numpy") else np.array(F1)
            return float(np.mean(arr))
        except Exception:
            continue
    return None


# ============================================================
# Eval cache and prediction
# ============================================================

_eval_cache: Dict[str, Any] = {}

def precompute_eval_predictions():
    if "pred_answers" in _eval_cache:
        return

    print("\n⚡ TEST → TRAIN full pipeline retrieval бір рет есептелуде...")
    t0 = time.time()

    pred_indices = []
    pred_q_sims = []
    pred_morph_sims = []
    pred_rerank_scores = []
    pred_answers = []

    n_test = len(test_q_emb)
    dense_topk = min(DENSE_TOPK, train_q_emb.shape[0])

    for start in range(0, n_test, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n_test)
        batch = test_q_emb[start:end]
        sims = batch @ train_q_emb.T

        local_top_idx = np.argpartition(-sims, kth=dense_topk - 1, axis=1)[:, :dense_topk]
        row_ids = np.arange(end - start)[:, None]
        local_top_vals = sims[row_ids, local_top_idx]

        order = np.argsort(-local_top_vals, axis=1)
        local_top_idx = np.take_along_axis(local_top_idx, order, axis=1)
        local_top_vals = np.take_along_axis(local_top_vals, order, axis=1)

        for r in range(end - start):
            global_i = start + r
            qv = batch[r]

            best = rerank_candidates_lite(
                q_emb=qv,
                q_tok_set=test_q_tok_sets[global_i],
                q_seg_set=test_q_seg_sets[global_i],
                cand_indices=local_top_idx[r].tolist(),
                cand_dense_scores=local_top_vals[r].tolist(),
                kb_q_tok_sets=train_q_tok_sets,
                kb_q_seg_sets=train_q_seg_sets,
                kb_a_emb=train_a_emb,
            )

            if best is None:
                pred_indices.append(-1)
                pred_q_sims.append(0.0)
                pred_morph_sims.append(0.0)
                pred_rerank_scores.append(0.0)
                pred_answers.append("Кешіріңіз, нақты жауап табылмады.")
            else:
                idx = int(best["idx"])
                ans = train_answers[idx]
                if POSTPROCESS_FOR_EVAL:
                    ans = answer_postprocess(ans)

                pred_indices.append(idx)
                pred_q_sims.append(float(best["dense_qsim"]))
                pred_morph_sims.append(float(best["morph_sim"]))
                pred_rerank_scores.append(float(best["rerank_score"]))
                pred_answers.append(ans)

    _eval_cache["pred_indices"] = np.array(pred_indices, dtype=np.int32)
    _eval_cache["pred_q_sims"] = np.array(pred_q_sims, dtype=np.float32)
    _eval_cache["pred_morph_sims"] = np.array(pred_morph_sims, dtype=np.float32)
    _eval_cache["pred_rerank_scores"] = np.array(pred_rerank_scores, dtype=np.float32)
    _eval_cache["pred_answers"] = pred_answers
    _eval_cache["true_answers"] = test_answers
    _eval_cache["test_questions"] = [clean_view(q) for q in test_questions]

    print(f"✅ TEST prediction дайын: {time.time() - t0:.2f} сек")

def _encode_answer_pairs_for_eval():
    if "pred_eval_emb" in _eval_cache and "true_eval_emb" in _eval_cache:
        return _eval_cache["pred_eval_emb"], _eval_cache["true_eval_emb"]

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n📐 Answer embedding-тері есептелуде...")
    t0 = time.time()

    pv = _encode_answers(preds, batch_size=ENCODE_BATCH_SIZE, normalize=True)
    tv = _encode_answers(trues, batch_size=ENCODE_BATCH_SIZE, normalize=True)

    _eval_cache["pred_eval_emb"] = pv
    _eval_cache["true_eval_emb"] = tv

    print(f"✅ Answer embedding дайын: {time.time() - t0:.2f} сек")
    return pv, tv


# ============================================================
# Metrics
# ============================================================

def evaluate_exact_at_1():
    precompute_eval_predictions()
    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    scores = [float(norm_for_exact(p) == norm_for_exact(t)) for p, t in zip(preds, trues)]
    return float(np.mean(scores))

def evaluate_token_f1_at_1():
    precompute_eval_predictions()
    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]
    scores = [token_f1(p, t) for p, t in zip(preds, trues)]
    return float(np.mean(scores))

def evaluate_mean_cos_qsim_at_1():
    precompute_eval_predictions()
    return float(np.mean(_eval_cache["pred_q_sims"]))

def evaluate_semantic_at_1_ans_cos(threshold: float = SEM_THR):
    pv, tv = _encode_answer_pairs_for_eval()
    ans_cos = np.sum(pv * tv, axis=1)
    _eval_cache["ans_cos"] = ans_cos
    return float(np.mean(ans_cos >= threshold))

def evaluate_bertscore_f1_at_1():
    if "bertscore_f1_mean" in _eval_cache:
        return float(_eval_cache["bertscore_f1_mean"])

    precompute_eval_predictions()

    preds = _eval_cache["pred_answers"]
    trues = _eval_cache["true_answers"]

    print("\n📊 BERTScoreF1@1 есептелуде...")
    t0 = time.time()

    f1_mean = _bert_lang_try(preds, trues)
    _eval_cache["bertscore_f1_mean"] = f1_mean

    print(f"✅ BERTScoreF1@1 дайын: {time.time() - t0:.2f} сек")
    return f1_mean

def build_eval_details():
    if "details_df" in _eval_cache:
        return _eval_cache["details_df"]

    precompute_eval_predictions()

    preds   = _eval_cache["pred_answers"]
    trues   = _eval_cache["true_answers"]
    qsims   = _eval_cache["pred_q_sims"]
    msims   = _eval_cache["pred_morph_sims"]
    rscores = _eval_cache["pred_rerank_scores"]

    if "ans_cos" not in _eval_cache:
        _ = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)

    ans_cos = _eval_cache["ans_cos"]
    rows = []

    for q, gold, pred, qsim, msim, rscore, acos in zip(
        _eval_cache["test_questions"], trues, preds, qsims, msims, rscores, ans_cos
    ):
        rows.append({
            "test_question": q,
            "gold_answer": gold,
            "pred_answer": pred,
            "QSim": float(qsim),
            "MorphSim": float(msim),
            "RerankScore": float(rscore),
            "Exact": float(norm_for_exact(pred) == norm_for_exact(gold)),
            "TokenF1": float(token_f1(pred, gold)),
            "AnsCos": float(acos),
            "SemHit": float(acos >= SEM_THR),
        })

    details_df = pd.DataFrame(rows)
    _eval_cache["details_df"] = details_df
    return details_df

def export_details_csv(path: str):
    details_df = build_eval_details()
    details_df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV exported: {path}  (rows={len(details_df)})")

def evaluate_system_full_schema():
    t0 = time.time()

    precompute_eval_predictions()

    exact_at_1    = evaluate_exact_at_1()
    token_f1_at_1 = evaluate_token_f1_at_1()
    mean_cos_qsim = evaluate_mean_cos_qsim_at_1()
    semantic_at_1 = evaluate_semantic_at_1_ans_cos(threshold=SEM_THR)
    bertscore_f1  = evaluate_bertscore_f1_at_1()

    total_t = time.time() - t0

    print("\n==================== RESULTS (FULL SCHEMA / HELD-OUT TEST) ====================")
    print("Dataset: FEMSeg + MAPP + Hybrid Retrieval + Lite Reranking + Answer Post-process")
    print(f"                     Exact@1: {exact_at_1:.6f}")
    print(f"                   TokenF1@1: {token_f1_at_1:.6f}")
    print(f"             MeanCos@1(QSim): {mean_cos_qsim:.6f}")
    print(f"    Semantic@1(ans_cos≥{SEM_THR}): {semantic_at_1:.6f}")

    if bertscore_f1 is not None:
        print(f"               BERTScoreF1@1: {bertscore_f1:.6f}")
    else:
        print("               BERTScoreF1@1: N/A (bert-score орнатылмаған)")

    print(f"               EvaluationTime: {total_t:.2f} sec")
    print(f"                    ALPHA_SEG: {ALPHA_SEG:.2f}")
    print(f"          HYBRID_DENSE_WEIGHT: {HYBRID_DENSE_WEIGHT:.2f}")
    print(f"          HYBRID_MORPH_WEIGHT: {HYBRID_MORPH_WEIGHT:.2f}")
    print(f"        RERANK_HYBRID_WEIGHT: {RERANK_HYBRID_WEIGHT:.2f}")
    print(f"           RERANK_ANS_WEIGHT: {RERANK_ANS_WEIGHT:.2f}")
    print(f"                    DENSE_TOPK: {DENSE_TOPK}")
    print(f"                   RERANK_TOPK: {RERANK_TOPK}")

    if EXPORT_CSV:
        export_details_csv(CSV_PATH)


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    try:
        while True:
            user_input = input("\nСұрақ енгізіңіз (шығу үшін 'exit'): ")
            if user_input.strip().lower() == "exit":
                print("Бағдарлама тоқтатылды. 👋")
                break

            (
                answer,
                idx,
                qsim,
                msim,
                rscore,
                q_clean,
                q_mapp,
                q_dense
            ) = ask_question_runtime(user_input, threshold=0.0)

            print("\n=== [FULL SCHEMA QA ЖАУАП] ===")
            print(answer)
            print(f"(idx={idx}, qsim={qsim:.4f}, morph={msim:.4f}, rerank={rscore:.4f})")

            print("\n--- Query views ---")
            print("clean :", q_clean)
            print("mapp  :", q_mapp)
            print("dense :", q_dense)

    except EOFError:
        pass

    evaluate_system_full_schema()

🖥 DEVICE: cuda
📥 QA JSON файлдан жүктеу: 50k.json
⚠ Ескерту: 256 жол оқылмады, бірақ қалғандары жүктелді.
📚 Барлық QA жазба саны: 50275
                                  question  \
0                     Python дегеніміз не?   
1           Python тілі қалай пайда болды?   
2  Python тілінің басты ерекшелігі қандай?   

                                              answer  
0  Python — қарапайым және оқуға оңай бағдарламал...  
1  Python тілін 1991 жылы Гвидо ван Россум жасаға...  
2  Python — оқуға жеңіл және интуитивті синтаксис...  
Mounted at /content/drive
⬇️ FEMSeg_v3 сегментаторын жүктеу...
✅ FEMSeg_v3 дайын.
🔎 FEMSeg тест: ▁Мен ▁мектеп ке ▁бар а мын
⬇️ MiniLM encoder-ін жүктеу: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ MiniLM моделі дайын.

🧠 MAPP нормализация басталды... (50275 жазба)
   MAPP 5000/48908 дайын | 8.0 сек
   MAPP 10000/48908 дайын | 12.7 сек
   MAPP 15000/48908 дайын | 14.7 сек
   MAPP 20000/48908 дайын | 16.7 сек
   MAPP 25000/48908 дайын | 18.9 сек
   MAPP 30000/48908 дайын | 20.4 сек
   MAPP 35000/48908 дайын | 22.1 сек
   MAPP 40000/48908 дайын | 24.5 сек
   MAPP 45000/48908 дайын | 29.4 сек
   MAPP 48908/48908 дайын | 33.3 сек

🧩 FEMSeg сегментация басталды... (50275 жазба)
   SEG 5000/48732 дайын | 3.1 сек
   SEG 10000/48732 дайын | 3.9 сек
   SEG 15000/48732 дайын | 4.7 сек
   SEG 20000/48732 дайын | 5.3 сек
   SEG 25000/48732 дайын | 5.8 сек
   SEG 30000/48732 дайын | 7.0 сек
   SEG 35000/48732 дайын | 7.5 сек
   SEG 40000/48732 дайын | 8.7 сек
   SEG 45000/48732 дайын | 11.4 сек
   SEG 48732/48732 дайын | 13.7 сек

🔎 Мысал:
                                   question  \
0                      Python дегеніміз не?   
1            Python тілі қалай пайда болды?   
2   Python т

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScoreF1@1 дайын: 54.29 сек

==================== RESULTS (FULL SCHEMA / HELD-OUT TEST) ====================
Dataset: FEMSeg + MAPP + Hybrid Retrieval + Lite Reranking + Answer Post-process
                     Exact@1: 0.015115
                   TokenF1@1: 0.474088
             MeanCos@1(QSim): 0.878803
    Semantic@1(ans_cos≥0.85): 0.379674
               BERTScoreF1@1: 0.835514
               EvaluationTime: 78.07 sec
                    ALPHA_SEG: 0.40
          HYBRID_DENSE_WEIGHT: 0.72
          HYBRID_MORPH_WEIGHT: 0.28
        RERANK_HYBRID_WEIGHT: 0.75
           RERANK_ANS_WEIGHT: 0.25
                    DENSE_TOPK: 80
                   RERANK_TOPK: 12

✅ CSV exported: qa_schema_hybrid_rerank_results.csv  (rows=5028)
